<a href="https://colab.research.google.com/github/Amper2B/GVC_MiniCaseStudy/blob/main/FusionModelv_0_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.notebook import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=== SYSTEM HARDWARE INITIALIZATION ===")
print(f"Active processing infrastructure destination: {device}\n")

class_names = ["boy", "corn", "coconut", "car", "monument",
               "dog", "face", "eye", "softdrink", "penguin"]

print("=== STEP 1: DIRECTORY VERIFICATION ===")
print("Using pre-uploaded images inside dataset/train/ and dataset/valid/.")

print("\n=== STEP 2: DATA PREPROCESSING PIPELINE ===")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class CustomImageFolder(ImageFolder):
    def find_classes(self, directory):
        classes = sorted(entry.name for entry in os.scandir(directory) if entry.is_dir())
        classes = [cls for cls in classes if cls != '.ipynb_checkpoints']
        if not classes:
            raise FileNotFoundError(f"Couldn't find any class folders in {directory}.")
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        return classes, class_to_idx

def is_image_file(path):
    if '.ipynb_checkpoints' in path:
        return False
    valid_extensions = ('.jpg', '.jpeg', '.png', '.ppm', '.bmp', '.pgm', '.tif', '.tiff', '.webp')
    return path.lower().endswith(valid_extensions)

try:
    train_dataset = CustomImageFolder('dataset/train', transform=train_transform, is_valid_file=is_image_file)
    val_dataset = CustomImageFolder('dataset/valid', transform=val_transform, is_valid_file=is_image_file)
except FileNotFoundError as e:
    print("Error: Target directories are unpopulated.")
    raise

num_classes = len(train_dataset.classes)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

print(f"Dynamic Categories Found : {train_dataset.classes}")
print(f"Training Samples Batched : {len(train_dataset)}")
print(f"Validation Samples Loaded: {len(val_dataset)}")

class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        self.resnet = models.resnet18(weights='DEFAULT')
        self.mobile = models.mobilenet_v2(weights='DEFAULT')
        self.resnet.fc = nn.Identity()
        self.mobile.classifier = nn.Identity()
        self.classifier = nn.Linear(1792, num_classes)

    def forward(self, x):
        f1 = self.resnet(x)
        f2 = self.mobile(x)
        combined = torch.cat((f1, f2), dim=1)
        return self.classifier(combined)

model = FusionModel(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_losses = []
val_losses = []

print("\n=== STEP 4: TRAINING CONCATENATED FUSION PIPELINE ===")
fusion_start = time.time()
for epoch in range(15):
    model.train()
    running_train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()

    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

    avg_train = running_train_loss / len(train_loader)
    avg_val = running_val_loss / len(val_loader)

    train_losses.append(avg_train)
    val_losses.append(avg_val)

    print(f"Epoch {epoch+1:02d}/15 | Train Loss: {avg_train:.4f} | Validation Loss: {avg_val:.4f}")
fusion_total_time = time.time() - fusion_start

plt.figure(figsize=(10, 5))
plt.plot(range(1, 16), train_losses, marker='o', color='blue', linewidth=2, label='Training Loss')
plt.plot(range(1, 16), val_losses, marker='s', color='red', linewidth=2, label='Validation Loss')
plt.title("Convergence Profile: Training vs. Validation Loss", fontsize=12, fontweight='bold')
plt.xlabel("Training Epoch", fontsize=10)
plt.ylabel("Cross-Entropy Loss", fontsize=10)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--')
plt.show()

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds, normalize='true')

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=train_dataset.classes, yticklabels=train_dataset.classes)
plt.title("Normalized Confusion Matrix (Validation Partition Data)", fontsize=12, fontweight='bold')
plt.ylabel('Actual Object Class', fontsize=10)
plt.xlabel('Predicted Object Class', fontsize=10)
plt.show()

print("\n=== CLASSIFICATION REPORT (VALIDATION PARTITION) ===")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))

baseline_model = models.resnet18(weights='DEFAULT')
baseline_model.fc = nn.Linear(512, num_classes)
baseline_model = baseline_model.to(device)
b_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=0.001)

print("\n=== STEP 7: TRAINING BASELINE DATA PIPELINE ===")
baseline_start = time.time()
for epoch in range(15):
    baseline_model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        b_optimizer.zero_grad()
        outputs = baseline_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        b_optimizer.step()
print("Baseline Training Complete.")
baseline_total_time = time.time() - baseline_start

base_acc = calculate_accuracy(baseline_model, val_loader)
fusion_acc = calculate_accuracy(model, val_loader)
improvement = fusion_acc - base_acc

plt.figure(figsize=(7, 5))
bars = plt.bar(['Baseline\n(ResNet18)', 'Proposed Fusion\n(ResNet18+MobileNetV2)'],
               [base_acc, fusion_acc], color=['darkgray', 'royalblue'], width=0.5)
plt.ylabel('Validation Accuracy (%)', fontsize=10)
plt.title('Empirical Performance Comparison', fontsize=12, fontweight='bold')
plt.ylim(0, 110)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')

plt.show()

print("\n=== EMPIRICAL VARIANCE REPORT ===")
print(f"Final Baseline Validation Accuracy: {base_acc:.2f}%")
print(f"Final Proposed Fusion Accuracy    : {fusion_acc:.2f}%")
print(f"Calculated Empirical Improvement   : {improvement:.2f}%")
print(f"Baseline Training Duration         : {baseline_total_time:.2f} seconds")
print(f"Proposed Fusion Training Duration  : {fusion_total_time:.2f} seconds\n")

print("=== STEP 10: LOCAL SERIALIZATION EXPORT ===")
torch.save(model.to('cpu').state_dict(), 'fusion_model_weights.pth')
files.download('fusion_model_weights.pth')
print("Model weights exported and downloaded successfully.")

=== SYSTEM HARDWARE INITIALIZATION ===
Active processing infrastructure destination: cpu

=== STEP 1: DIRECTORY VERIFICATION ===
Using pre-uploaded images inside dataset/train/ and dataset/valid/.

=== STEP 2: DATA PREPROCESSING PIPELINE ===
Dynamic Categories Found : ['boy', 'car', 'coconut', 'corn', 'dog', 'eye', 'face', 'monument', 'penguin', 'softdrink']
Training Samples Batched : 1262
Validation Samples Loaded: 315

=== STEP 4: TRAINING CONCATENATED FUSION PIPELINE ===
